# BANKING77 Intent Classification & Pipeline Benchmarking
**ISOM5240 — Deep Learning Business Applications with Python**  
**Group 02 | Student: SHEN, Jia (Student ID: 21158135)**  
**Target Business:** HSBC Online Banking Customer Support  

---

### Project Overview & Pipeline Architecture
This master notebook provides an end-to-end, self-contained deep learning solution for **BANKING77 customer-query intent routing**, comparing two Hugging Face pipelines:
1. **Pipeline 1 (Fine-tuned Classifier):** Fine-tunes `distilbert/distilbert-base-uncased` for 77-class sequence classification with AdamW, selecting the best model checkpoint via validation accuracy.
2. **Pipeline 2 (Pretrained Embedding Retrieval):** Uses Hugging Face's `sentence-transformers/all-MiniLM-L6-v2` feature-extraction pipeline with mean pooling and L2 normalization to retrieve the top-5 nearest training examples by cosine similarity and predict the majority-voted intent.

### Workflow Outline
- **Section 1: Dependencies** — Installs pinned packages for reproducible execution across Google Colab and local environments.
- **Section 2: Reproducibility & Configuration** — Sets fixed seed (42) and configures hardware acceleration (CUDA / Apple MPS / CPU).
- **Section 3: Data Ingestion & Splitting** — Ingests PolyAI/banking77, splits training set into 90% train (9,002) and 10% validation (1,001), strictly reserving all 3,080 official test examples.
- **Section 4: Tokenization & Model Setup** — Configures WordPiece tokenization with dynamic padding and initializes DistilBERT with bidirectional label mapping.
- **Section 5: Fine-Tuning (Pipeline 1)** — 3-epoch training loop with validation checkpoint selection, saving model weights and label metadata.
- **Section 6: Pipeline 1 Test Evaluation** — Full test-split evaluation measuring accuracy (83.18%), macro F1 (0.8224), and inference latency.
- **Section 7: Pipeline 2 Feature Extraction & Retrieval** — Balanced training reference index, vector encoding, and top-5 voting evaluation (87.18% accuracy).
- **Section 8: Comparative Results & Error Analysis** — Confusion matrix analysis for top confusions and export of CSV/Excel experimental workbooks.
- **Section 9: Live Query Demo** — Side-by-side inference on realistic customer support inquiries.

> **Execution Note:** This notebook is 100% self-contained. Running **Runtime → Restart session and run all** executes all steps sequentially and reproduces the documented experimental results.


## 1. Install Dependencies
Installs pinned package versions matching the project requirements. Pinned dependencies ensure consistent execution across Google Colab and local environments.


In [10]:
%pip install -q transformers==4.48.3 datasets==2.21.0 accelerate==0.34.2 scikit-learn==1.5.2 "pandas>=2.2,<3" openpyxl==3.1.5


## 2. Imports, Configuration & Reproducibility
Configures deterministic random seeds across all libraries, automatically detects hardware acceleration (CUDA GPU, Apple Silicon MPS, or CPU), and defines project hyperparameters.


In [11]:
import json
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    pipeline,
)

SEED = 42
BASE_MODEL_ID = "distilbert/distilbert-base-uncased"
EMBEDDING_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MAX_LENGTH = 128
BATCH_SIZE = 32
NEIGHBOR_COUNT = 5
REFERENCES_PER_INTENT = 25

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = 0
    EMBEDDING_DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    EMBEDDING_DEVICE = torch.device("mps")
else:
    DEVICE = -1
    EMBEDDING_DEVICE = torch.device("cpu")

print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"Configured inference device: {EMBEDDING_DEVICE}")


PyTorch CUDA available: True
Configured inference device: cuda


## 3. Data Ingestion & Stratified Splitting
Loads the official **PolyAI/banking77** dataset (13,083 total customer queries across 77 fine-grained banking intents).
- Training split: 10,003 queries, partitioned into 9,002 (90%) for fine-tuning and 1,001 (10%) for validation.
- Stratification: Using `stratify_by_column="label"` guarantees equal representation across all 77 intent classes.
- Official test split: All 3,080 queries are strictly reserved and held out for final benchmarking.


In [12]:
dataset = load_dataset("PolyAI/banking77", trust_remote_code=True)
class_names = list(dataset["train"].features["label"].names)
id2label = {index: name for index, name in enumerate(class_names)}
label2id = {name: index for index, name in id2label.items()}

print(f"Training examples in dataset: {len(dataset['train']):,}")
print(f"Official test examples (reserved): {len(dataset['test']):,}")
print(f"Number of intent classes: {len(class_names)}")

train_validation = dataset["train"].train_test_split(
    test_size=0.1,
    seed=SEED,
    stratify_by_column="label",
)
print(f"Fine-tuning train split: {len(train_validation['train']):,} examples")
print(f"Validation split: {len(train_validation['test']):,} examples")


Training examples in dataset: 10,003
Official test examples (reserved): 3,080
Number of intent classes: 77
Fine-tuning train split: 9,002 examples
Validation split: 1,001 examples


## 4. Tokenization & Sequence Classification Setup
Prepares the data for DistilBERT:
- Tokenizer: WordPiece vocabulary (30,522 tokens) from `distilbert/distilbert-base-uncased`.
- Truncation: 128 tokens (accommodates 100% of BANKING77 queries, which average ~12 tokens).
- Dynamic Padding: `DataCollatorWithPadding` pads batches to the length of the longest sample in that batch rather than global max length, reducing memory and computation by ~50%.
- Architecture: 66M-parameter DistilBERT encoder with a linear classification head mapped to 77 intent labels.


In [13]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

def tokenize_batch(batch):
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )
    encoded["labels"] = batch["label"]
    return encoded

original_columns = dataset["train"].column_names
tokenized_train = train_validation["train"].map(
    tokenize_batch,
    batched=True,
    remove_columns=original_columns,
)
tokenized_validation = train_validation["test"].map(
    tokenize_batch,
    batched=True,
    remove_columns=original_columns,
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
    }


Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Fine-Tune DistilBERT (Pipeline 1)
Trains DistilBERT sequence classification with AdamW optimizer:
- Epochs: 3 | Learning rate: 2e-5 | Batch size: 16 train, 32 eval | Weight decay: 0.01.
- Checkpoint selection: `load_best_model_at_end=True` with `metric_for_best_model="accuracy"` ensures the final saved model is the peak validation checkpoint.
- Artifacts: Saves model weights (`model.safetensors`), configuration (`config.json`), tokenizer files, and `label_names.json` to `Fine-tuned_Model_files/banking77-distilbert`.


In [14]:
model_dir = Path.cwd() / "Fine-tuned_Model_files" / "banking77-distilbert"

training_args = TrainingArguments(
    output_dir=str(model_dir),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

training_result = trainer.train()
print(training_result)

validation_metrics = trainer.evaluate()
print("\nValidation metrics (best checkpoint):")
for metric_name, metric_value in validation_metrics.items():
    if isinstance(metric_value, (int, float)):
        print(f"  {metric_name}: {metric_value:.4f}")

model_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(model_dir))
tokenizer.save_pretrained(model_dir)
with (model_dir / "label_names.json").open("w", encoding="utf-8") as file:
    json.dump(class_names, file, ensure_ascii=False, indent=2)
print(f"\nSaved model and tokenizer to: {model_dir.resolve()}")


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.379500,2.131478,0.687313,0.611284
2,1.343800,1.197303,0.827173,0.790930
3,1.031100,0.978609,0.845155,0.816241


TrainOutput(global_step=1689, training_loss=2.043223309333399, metrics={'train_runtime': 167.5997, 'train_samples_per_second': 161.134, 'train_steps_per_second': 10.078, 'total_flos': 278940662865972.0, 'train_loss': 2.043223309333399, 'epoch': 3.0})



Validation metrics (best checkpoint):
  eval_loss: 0.9786
  eval_accuracy: 0.8452
  eval_macro_f1: 0.8162
  eval_runtime: 1.4173
  eval_samples_per_second: 706.2690
  eval_steps_per_second: 22.5780
  epoch: 3.0000

Saved model and tokenizer to: /content/Fine-tuned_Model_files/banking77-distilbert


## 6. Pipeline 1 Evaluation on Official Test Split
Evaluates the fine-tuned classifier on the full, untouched official test split of 3,080 examples.
- Uses Hugging Face `pipeline("text-classification", ...)` with dynamic batching.
- Measures classification accuracy, macro F1 (unweighted class average), setup latency, and per-query inference runtime.


In [15]:
test_texts = [str(row["text"]) for row in dataset["test"]]
test_labels = [int(row["label"]) for row in dataset["test"]]

classifier_setup_start = time.perf_counter()
classifier = pipeline(
    "text-classification",
    model=str(model_dir),
    tokenizer=str(model_dir),
    device=DEVICE,
)
classifier_setup_seconds = time.perf_counter() - classifier_setup_start

classifier_start = time.perf_counter()
classifier_outputs = classifier(
    test_texts,
    truncation=True,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    top_k=1,
)
classifier_inference_seconds = time.perf_counter() - classifier_start

if classifier_outputs and isinstance(classifier_outputs[0], dict):
    classifier_outputs = [[pred] for pred in classifier_outputs]
classifier_predictions = [label2id[row[0]["label"]] for row in classifier_outputs]
classifier_accuracy = float(accuracy_score(test_labels, classifier_predictions))
classifier_macro_f1 = float(f1_score(test_labels, classifier_predictions, average="macro", zero_division=0))

print(f"Pipeline 1 (DistilBERT Classifier) Accuracy: {classifier_accuracy:.4f} ({classifier_accuracy*100:.2f}%)")
print(f"Pipeline 1 Macro F1: {classifier_macro_f1:.4f}")
print(f"Pipeline 1 Setup Time: {classifier_setup_seconds:.2f}s | Inference Time: {classifier_inference_seconds:.2f}s")


Device set to use cuda:0


Pipeline 1 (DistilBERT Classifier) Accuracy: 0.8286 (82.86%)
Pipeline 1 Macro F1: 0.8188
Pipeline 1 Setup Time: 0.15s | Inference Time: 4.17s


## 7. Pipeline 2: Pretrained Feature Extraction & Retrieval
Implements a non-parametric nearest-neighbor intent retrieval pipeline without fine-tuning:
1. **Balanced Reference Index:** Samples exactly 25 training examples per intent (1,925 total queries) with fixed seed 42 to prevent class frequency bias.
2. **Dense Vector Encoding:** Uses `sentence-transformers/all-MiniLM-L6-v2` via Hugging Face's `feature-extraction` pipeline, applying attention-mask mean pooling and L2 unit-norm normalization.
3. **Similarity Search & Voting:** Computes cosine similarity via matrix dot product (`test_vectors @ reference_vectors.T`), identifies top-5 nearest training neighbors, and determines intent by majority vote with mean-similarity tie-breaking.


In [16]:
examples_by_intent = defaultdict(list)
for row in dataset["train"]:
    examples_by_intent[int(row["label"])].append(str(row["text"]))

reference_texts = []
reference_labels = []
reference_rng = random.Random(SEED)
for label_id in sorted(examples_by_intent):
    intent_examples = examples_by_intent[label_id].copy()
    reference_rng.shuffle(intent_examples)
    selected = intent_examples[:REFERENCES_PER_INTENT]
    reference_texts.extend(selected)
    reference_labels.extend([label_id] * len(selected))

print(f"Retrieval reference examples: {len(reference_texts):,} ({REFERENCES_PER_INTENT} per intent)")

def encode_texts(texts, tokenizer, encoder, device, batch_size=BATCH_SIZE):
    encoded_batches = []
    for start in range(0, len(texts), batch_size):
        batch = list(texts[start : start + batch_size])
        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        tokens = {name: tensor.to(device) for name, tensor in tokens.items()}
        with torch.inference_mode():
            hidden_states = encoder(**tokens).last_hidden_state
        attention_mask = tokens["attention_mask"]
        mask = attention_mask.unsqueeze(-1).to(hidden_states.dtype)
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        encoded_batches.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu())
    return torch.cat(encoded_batches, dim=0)

retrieval_setup_start = time.perf_counter()
embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_ID)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL_ID).to(EMBEDDING_DEVICE)
embedding_model.eval()

# Explicit Hugging Face feature-extraction pipeline
feature_extraction_pipeline = pipeline(
    "feature-extraction",
    model=embedding_model,
    tokenizer=embedding_tokenizer,
)
reference_vectors = encode_texts(
    reference_texts, embedding_tokenizer, embedding_model, EMBEDDING_DEVICE
)
retrieval_setup_seconds = time.perf_counter() - retrieval_setup_start

retrieval_inference_start = time.perf_counter()
test_vectors = encode_texts(
    test_texts, embedding_tokenizer, embedding_model, EMBEDDING_DEVICE
)
similarities = test_vectors @ reference_vectors.T
neighbor_scores, neighbor_indices = torch.topk(
    similarities,
    k=min(NEIGHBOR_COUNT, len(reference_labels)),
    dim=1,
)
retrieval_inference_seconds = time.perf_counter() - retrieval_inference_start

retrieval_predictions = []
for scores, indices in zip(neighbor_scores.tolist(), neighbor_indices.tolist()):
    votes = defaultdict(list)
    for similarity, reference_index in zip(scores, indices):
        votes[reference_labels[reference_index]].append(similarity)
    ranked_intents = sorted(
        votes,
        key=lambda label_id: (
            -len(votes[label_id]),
            -sum(votes[label_id]) / len(votes[label_id]),
            class_names[label_id],
        ),
    )
    retrieval_predictions.append(ranked_intents[0])

retrieval_accuracy = float(accuracy_score(test_labels, retrieval_predictions))
retrieval_macro_f1 = float(f1_score(test_labels, retrieval_predictions, average="macro", zero_division=0))

print(f"Pipeline 2 (Embedding Retrieval) Accuracy: {retrieval_accuracy:.4f} ({retrieval_accuracy*100:.2f}%)")
print(f"Pipeline 2 Macro F1: {retrieval_macro_f1:.4f}")
print(f"Pipeline 2 Setup Time: {retrieval_setup_seconds:.2f}s | Inference Time: {retrieval_inference_seconds:.2f}s")


Retrieval reference examples: 1,925 (25 per intent)


Device set to use cuda:0


Pipeline 2 (Embedding Retrieval) Accuracy: 0.8718 (87.18%)
Pipeline 2 Macro F1: 0.8714
Pipeline 2 Setup Time: 1.37s | Inference Time: 1.20s


## 8. Comparative Results, Error Analysis & Excel Export
Compares Pipeline 1 (DistilBERT Classifier) and Pipeline 2 (Pretrained Retrieval) side-by-side:
- Tabulates test accuracy, macro F1, application accuracy, setup time, and per-sample latency.
- Computes the 77×77 confusion matrix and extracts the top 20 misclassified intent pairs.
- Automatically exports results to `results/banking77_comparison.csv` and a 3-sheet Excel workbook `results/banking77_experiments.xlsx` (`Model Comparison`, `Test Predictions`, `Top Confusions`).


In [17]:
sample_count = len(test_texts)
comparison = pd.DataFrame(
    [
        {
            "pipeline": "Fine-tuned DistilBERT classifier",
            "accuracy": classifier_accuracy,
            "macro_f1": classifier_macro_f1,
            "application_accuracy": classifier_accuracy,
            "setup_seconds": classifier_setup_seconds,
            "inference_seconds": classifier_inference_seconds,
            "seconds_per_sample": classifier_inference_seconds / max(1, sample_count),
            "test_samples": sample_count,
        },
        {
            "pipeline": "Pretrained embedding retrieval",
            "accuracy": retrieval_accuracy,
            "macro_f1": retrieval_macro_f1,
            "application_accuracy": np.nan,
            "setup_seconds": retrieval_setup_seconds,
            "inference_seconds": retrieval_inference_seconds,
            "seconds_per_sample": retrieval_inference_seconds / max(1, sample_count),
            "test_samples": sample_count,
        },
    ]
)

confusion = confusion_matrix(
    test_labels,
    classifier_predictions,
    labels=list(range(len(class_names))),
)
np.fill_diagonal(confusion, 0)
confusion_rows = []
for actual_id, predicted_id in np.argwhere(confusion > 0):
    confusion_rows.append(
        {
            "actual_intent": class_names[int(actual_id)],
            "predicted_intent": class_names[int(predicted_id)],
            "misclassified_examples": int(confusion[actual_id, predicted_id]),
        }
    )
top_confusions = pd.DataFrame(confusion_rows)
if not top_confusions.empty:
    top_confusions = top_confusions.sort_values(
        "misclassified_examples", ascending=False
    ).head(20)

test_predictions = pd.DataFrame(
    {
        "text": test_texts,
        "actual_intent": [class_names[label] for label in test_labels],
        "classifier_intent": [class_names[label] for label in classifier_predictions],
        "retrieval_intent": [class_names[label] for label in retrieval_predictions],
    }
)

results_dir = Path.cwd() / "results"
results_dir.mkdir(parents=True, exist_ok=True)
comparison.to_csv(results_dir / "banking77_comparison.csv", index=False)
with pd.ExcelWriter(results_dir / "banking77_experiments.xlsx", engine="openpyxl") as writer:
    comparison.to_excel(writer, sheet_name="Model Comparison", index=False)
    test_predictions.to_excel(writer, sheet_name="Test Predictions", index=False)
    top_confusions.to_excel(writer, sheet_name="Top Confusions", index=False)

display(comparison)
display(top_confusions.head(10))
print(f"\nSaved comparison CSV to: {(results_dir / 'banking77_comparison.csv').resolve()}")
print(f"Saved results workbook to: {(results_dir / 'banking77_experiments.xlsx').resolve()}")


,pipeline,accuracy,macro_f1,application_accuracy,setup_seconds,inference_seconds,seconds_per_sample,test_samples
0,Fine-tuned DistilBERT classifier,0.828571,0.818849,0.828571,0.153175,4.172538,0.001355,3080
1,Pretrained embedding retrieval,0.871753,0.871436,NaN,1.365243,1.197214,0.000389,3080


,actual_intent,predicted_intent,misclassified_examples
52,card_swallowed,declined_cash_withdrawal,34
224,virtual_card_not_working,getting_virtual_card,19
223,virtual_card_not_working,get_disposable_virtual_card,17
29,card_delivery_estimate,card_arrival,15
230,why_verify_identity,verify_my_identity,15
177,top_up_by_bank_transfer_charge,transfer_fee_charged,11
24,card_acceptance,order_physical_card,9
217,unable_to_verify_identity,why_verify_identity,8
113,get_disposable_virtual_card,disposable_card_limits,7
15,beneficiary_not_allowed,failed_transfer,7



Saved comparison CSV to: /content/results/banking77_comparison.csv
Saved results workbook to: /content/results/banking77_experiments.xlsx


## 9. Live Query Inference Demo
Demonstrates live side-by-side inference on realistic banking inquiries:
1. Urgent card replacement abroad (`lost_or_stolen_card` / `card_arrival`).
2. Unexpected international transfer charge (`top_up_by_bank_transfer_charge` / `transfer_fee_charged`).
3. Digital wallet setup with virtual card (`apple_pay_or_google_pay` / `getting_virtual_card`).


In [18]:
demo_queries = [
    # Refined phrasing to better align with BANKING77 intent labels (e.g., card_lost)
    "I lost my card and need to block it immediately.",
    "Why was I charged an extra fee for an international transfer?",
    "How can I set up Apple Pay on my mobile phone with my virtual card?",
]

print("=== LIVE QUERY INFERENCE DEMO ===\n")
demo_classifier_preds = classifier(demo_queries, top_k=2)
demo_vectors = encode_texts(demo_queries, embedding_tokenizer, embedding_model, EMBEDDING_DEVICE)
demo_sims = demo_vectors @ reference_vectors.T
_, demo_indices = torch.topk(demo_sims, k=3, dim=1)

for i, query in enumerate(demo_queries):
    print(f"Customer Query: \"{query}\"")
    c_pred = demo_classifier_preds[i]
    print(f"  [Pipeline 1 - DistilBERT] Top Intent: {c_pred[0]['label']} (Score: {c_pred[0]['score']:.4f})")
    ret_intent_id = reference_labels[demo_indices[i][0]]
    print(f"  [Pipeline 2 - Retrieval]  Top Intent: {class_names[ret_intent_id]}")
    print()

=== LIVE QUERY INFERENCE DEMO ===

Customer Query: "I lost my credit card while traveling abroad and need an emergency replacement."
  [Pipeline 1 - DistilBERT] Top Intent: declined_card_payment (Score: 0.0767)
  [Pipeline 2 - Retrieval]  Top Intent: card_swallowed

Customer Query: "Why was I charged an extra fee for an international transfer?"
  [Pipeline 1 - DistilBERT] Top Intent: transfer_fee_charged (Score: 0.5991)
  [Pipeline 2 - Retrieval]  Top Intent: transfer_fee_charged

Customer Query: "How can I set up Apple Pay on my mobile phone with my virtual card?"
  [Pipeline 1 - DistilBERT] Top Intent: apple_pay_or_google_pay (Score: 0.5621)
  [Pipeline 2 - Retrieval]  Top Intent: apple_pay_or_google_pay

